# 02-5. 勾配・ヤコビアン・ヘッシアン — 動かして確かめる

📖 解説: [`../05_gradient_jacobian.md`](../05_gradient_jacobian.md)

## このノートで触るもの
1. 勾配 ∇f を JAX で
2. ヤコビアン Jf を可視化
3. ヘッシアン Hf と固有値で点の性質を判定 (最小・最大・鞍点)
4. VJP / JVP で大規模対応

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (05_gradient_jacobian.md)](../05_gradient_jacobian.md)

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

%matplotlib inline

## 1. 勾配 ∇f

In [ ]:
def f(p):
    return p[0]**2 + 2*p[0]*p[1] + 3*p[1]**2

grad_f = jax.grad(f)

for point in [[0, 0], [1, 1], [-2, 3]]:
    p = jnp.array(point, dtype=float)
    print(f'∇f at {point}: {grad_f(p)}')

## 2. ヤコビアン Jf

In [ ]:
def f_vec(p):
    """f: ℝ² → ℝ³."""
    return jnp.array([p[0] + p[1], p[0] * p[1], p[0]**2])

J_rev = jax.jacrev(f_vec)
J_fwd = jax.jacfwd(f_vec)

p = jnp.array([1.0, 2.0])
print('jacrev:')
print(J_rev(p))
print('jacfwd:')
print(J_fwd(p))
print('\n→ 同じ結果。出力次元 < 入力次元なら jacrev、逆なら jacfwd が高速')

## 3. ヘッシアン Hf と最小・最大・鞍点の判定

In [ ]:
test_funcs = {
    '最小点 (お椀)': lambda p: p[0]**2 + p[1]**2,
    '最大点 (山)':   lambda p: -(p[0]**2 + p[1]**2),
    '鞍点 (馬の鞍)': lambda p: p[0]**2 - p[1]**2,
}

for name, f in test_funcs.items():
    H = jax.hessian(f)(jnp.array([0.0, 0.0]))
    eigs = jnp.linalg.eigvalsh(H)
    print(f'{name}:')
    print(f'  Hessian = \n{np.array(H)}')
    print(f'  固有値: {np.array(eigs)}')
    if jnp.all(eigs > 0):
        verdict = '✅ 全部正 → 最小点'
    elif jnp.all(eigs < 0):
        verdict = '✅ 全部負 → 最大点'
    else:
        verdict = '⚠️  正負混在 → 鞍点'
    print(f'  判定: {verdict}\n')

## 4. 鞍点を 3D で可視化

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

x = np.linspace(-2, 2, 50)
y = np.linspace(-2, 2, 50)
X, Y = np.meshgrid(x, y)
Z_min = X**2 + Y**2          # 最小
Z_max = -(X**2 + Y**2)       # 最大
Z_saddle = X**2 - Y**2       # 鞍点

fig = plt.figure(figsize=(15, 5))
for i, (Z, title) in enumerate([(Z_min, '最小: x²+y²'),
                                 (Z_max, '最大: -(x²+y²)'),
                                 (Z_saddle, '鞍点: x²-y²')], 1):
    ax = fig.add_subplot(1, 3, i, projection='3d')
    ax.plot_surface(X, Y, Z, cmap='coolwarm', alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout()
plt.show()

## 5. VJP — 大規模ニューラルネットの基本

In [ ]:
# 入力 100 次元、出力 10 次元の関数
rng = jax.random.PRNGKey(0)
W = jax.random.normal(rng, shape=(10, 100))

def f(x):
    return jnp.tanh(W @ x)

x = jax.random.normal(jax.random.PRNGKey(1), shape=(100,))
y, vjp_fn = jax.vjp(f, x)

# 出力空間の "勾配" v に対し、入力空間の勾配を返す
v = jnp.ones(10)
(grad_x,) = vjp_fn(v)
print(f'入力 shape: {x.shape}')
print(f'出力 shape: {y.shape}')
print(f'伝わった入力勾配 shape: {grad_x.shape}')
print('\n→ 完全なヤコビアン (10×100=1000 要素) を作らずに済む。\n   1750億パラメータでも同じ仕組みでメモリ効率良くいける。')

## 6. ML 訓練の本体 — `value_and_grad`

In [ ]:
# 線形回帰: y = wx + b
def loss(params, x, y):
    w, b = params
    return jnp.mean((y - (w * x + b))**2)

# 値と勾配を同時に取得 (両方欲しい場合の定石)
value_and_grad = jax.value_and_grad(loss)

# データ
x_data = jnp.linspace(0, 10, 100)
y_data = 2 * x_data + 1 + 0.1 * jax.random.normal(rng, shape=(100,))

params = (0.0, 0.0)  # (w, b)
lr = 0.01

print('epoch  loss      w       b')
for epoch in range(20):
    val, grads = value_and_grad(params, x_data, y_data)
    params = (params[0] - lr * grads[0], params[1] - lr * grads[1])
    if epoch % 4 == 0:
        print(f'{epoch:5d}  {float(val):.4f}  {float(params[0]):.4f}  {float(params[1]):.4f}')

print(f'\n最終 w, b: {float(params[0]):.4f}, {float(params[1]):.4f}  (真値 2.0, 1.0)')

## まとめ
- `jax.grad` でスカラー関数の勾配 (ベクトル)
- `jax.jacrev` / `jacfwd` でベクトル関数のヤコビアン (行列)
- `jax.hessian` でスカラー関数のヘッセ行列 (固有値で点の種類が判定可)
- `jax.vjp` で大規模パラメータでもメモリ効率良く勾配が取れる
- `jax.value_and_grad` で訓練ループの定型を 1 行に

## 微積分章 完了 🎉

→ 次の章: [`../../03_probability_statistics/README.md`](../../03_probability_statistics/README.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次の章 → |
|---|---|---|---|
| [`04_multivariable.ipynb`](04_multivariable.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`../../03_probability_statistics/README.md`](../../03_probability_statistics/README.md) |